In [0]:
from pyspark.sql.functions import *
from datetime import datetime
import os
import logging
import psycopg2

#==============================
# Widgets
#==============================

parameters = { "user" : "", 
               "password" : "",
               "driver":"",
               "jdbc_url":"",
               "outsourcepath":"",
               "logspath":""}

for k,v in parameters.items():
    dbutils.widgets.text(k,v)

user = dbutils.widgets.get("user")
password = dbutils.widgets.get("password")
driver = dbutils.widgets.get("driver")
jdbc_url = dbutils.widgets.get("jdbc_url")
outsourcepath = dbutils.widgets.get("outsourcepath")
logspath = dbutils.widgets.get("logspath")

#===============================================
#Auditing logs
#===============================================
def auditing_employee_extract_audit(
    rows,
    status,
    message
):
    conn = psycopg2.connect(
        host="ep-calm-math-d8p0rx76.database.us-east-2.cloud.databricks.com",
        port="5432",
        database="databricks_postgres",
        user="testdb",
        password="npg_ejDE7ikJbd5m"
    )

    cursor = conn.cursor()

    # CALL procedure
    cursor.execute(
        """
        CALL sp_employee_extract_audit(
            %s::INT,
            %s::VARCHAR,
            %s::VARCHAR,
            %s::VARCHAR
        )
        """,
        (rows, status, message, '')
    )

    # Fetch INOUT result
    result = cursor.fetchone()
    conn.commit()

    cursor.close()
    conn.close()
    
    return result
#===============================================
# Validations
#===============================================

if not user:
    raise Exception("User is empty")

if not password:
    raise Exception("Password is empty")

if not driver:
    raise Exception("Driver is empty")

if not jdbc_url:
    raise Exception("JDBC URL is empty")

if not outsourcepath:
    raise Exception("Outsource path is empty")

if not logspath:
    raise Exception("Logs path is empty")

#=====================================================
# Logging Configuration
#=====================================================

os.makedirs(logspath,exist_ok = True)

logs_file_name = os.path.join(logspath,
                            f"{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}_employees_extract.log")

logger = logging.getLogger()

for h in logger.handlers[:]:
    logger.removeHandler(h)

logging.basicConfig(
    filename= logs_file_name,
    level = logging.DEBUG,
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

#=====================================================
# Actual Logic
#=====================================================

try:
    logger.info("Job Starts")
    logger.info("Extracting employees data from postgresql database")
    logger.debug(f"user: {user}")
    logging.debug(f"password: {password}")
    logging.debug(f"driver: {driver}")
    logging.debug(f"jdbc_url: {jdbc_url}")
    logging.debug(f"outsourcepath: {outsourcepath}")
    logging.debug(f"logspath: {logspath}")


    connection_properties = {
        "user": user,
        "password": password,
        "driver": driver
    }

    df = spark.read.jdbc(
        url=jdbc_url,
        table="employees",
        properties=connection_properties
    )

    logger.info("Extracting only today's changed or inserted records")

    today_records = df.filter("to_date(last_updated_date) = current_date()")

    logging.debug(f"today_records count: {today_records.count()}")

    today_records = today_records.select(
        *[c for c in today_records.columns if c != "last_updated_date"]
    )


    if today_records.count() > 0:
        logging.info("Found changed or inserted records for today")
        logging.info("Writing to csv file")
        
        today_records.coalesce(1).write \
        .option("header",True)\
        .format("csv") \
        .mode("append") \
        .save(outsourcepath)

        logging.debug(f"File was written successfully in {outsourcepath}")

        logging.info("Calling the audit procedure")
        result = auditing_employee_extract_audit(
        today_records.count(),
        "Success",
        " ")
        logging.debug(f"Result: {result}")
                      
        logging.info("Renaming the file in the businesss specified format")

        files = dbutils.fs.ls(outsourcepath)
        files = [f for f in files if f.path.endswith(".csv")]
        if files:
            for file in files:
                today_date = datetime.today().strftime("%Y%m%d")
                file_name = f"employees_{today_date}.csv"
                new_file_name = os.path.join(outsourcepath,file_name)
                dbutils.fs.mv(file.path,new_file_name)
                logging.debug(f"File was renamed successfully to {new_file_name}")
        else:
            logging.error("No file was found to rename")
    else:
        logging.info("No changed or inserted records for today")
        logging.info("Calling the audit procedure")
        result = auditing_employee_extract_audit(
        0,
        "Success",
        "No changed or inserted records for today")
        logging.debug(f"Result: {result}")

except Exception as e:
    logger.exception(f"Error: {str(e)}")
    result = auditing_employee_extract_audit(
        0,
        "Failed",
        f"Error: {str(e)}")
    
    logging.debug(f"Result: {result}")

finally:

    logging.info("Job Ended")

    logging.shutdown()

    print(f"Log File Created: {logs_file_name}")
